# Comparación de resultados reporte bibliométrico manual vs automática
Para Honduras y Ecuador.

## 0. Configuración y funciones auxiliares

Se trabaja sobre copias de las cuatro bases leídas arriba. Además se cargan, solo para lectura:
- `data/queries_<país>.json`: términos de país y palabras clave de cada eje.
- `data/subfields_tagged_<país>.xlsx`: subcampos de interés.
- Hoja `topics` de `outputs/bibliometrics_<país>_db.xlsx`: tópico → subcampo.
- `outputs/bibliometrics_<país>_backup/`: resumen de la corrida (`run_summary.json`) y respaldos crudos de cada consulta (`thematic_*.json`), antes del filtro de subcampo y de la eliminación de duplicados.

`FECHA_MANUAL` es la fecha de la portada de cada reporte manual. `BRUTOS_MANUAL` transcribe los resultados brutos que informan los PDF:
- Honduras: por objetivo estratégico, págs. 8–13.
- Ecuador: por eje temático, págs. 4–10.

In [1]:
import glob
import json
import re
import unicodedata
import difflib
import pandas as pd

from IPython.display import display, Markdown

pd.set_option('display.max_colwidth', 110)
pd.set_option('display.max_rows', 150)
pd.set_option('display.width', 250)

# Bases planas
hon_aut = pd.read_excel("../outputs/bibliometrics_honduras.xlsx")
hon_man = pd.read_excel("../docs/Honduras 2019 - 2026 con ejes tematicos 1406.xlsx",
    sheet_name="Honduras 2019-2026")

ecu_aut = pd.read_excel("../outputs/bibliometrics_ecuador.xlsx")
ecu_man = pd.read_excel("../docs/ECUADOR _ FINAL 11595 publicaciones.xlsx", 
    sheet_name="Master final file")

TIPOS = ['article', 'report', 'dissertation', 'book', 'book-chapter', 'conference-paper', 'dataset', 'data-paper']

PAISES = {
    'Honduras': dict(man=hon_man.copy(), aut=hon_aut.copy(), slug='honduras', fecha_manual='2026-07-01',
                     titulo='Título', anio='Año', tipo='Tipo', abstract='Abstract', citas='Cited by',
                     oe='Objetivo estratégico único (OE)', idioma=None, llave='DOI', nombre_llave='doi'),
    'Ecuador': dict(man=ecu_man.copy(), aut=ecu_aut.copy(), slug='ecuador', fecha_manual='2026-10-24',
                    titulo='Título', anio='Año', tipo='Tipo', abstract='Abstracto', citas='Cited by',
                    oe='Objetivo Estratégico (OE)', idioma='Idioma', llave='OpenAlex ID ', nombre_llave='openalex_id'),
}

BRUTOS_MANUAL = {
    'Honduras': {'1.1': 99, '1.2': 698, '2.1': 43, '2.2': 285, '3.1': 48, '3.2': 62, '3.3': 286},
    'Ecuador': {'1111': 3247, '1112': 798, '1113': 148, '1121': 362, '1122': 417, '1131': 1138, '1141': 405,
                '1142': 101, '1211': 949, '1221': 139, '2111': 1632, '2121': 419, '2131': 842, '3111': 414,
                '3112': 511, '3113': 129, '3121': 806, '3122': 36, '3211': 163, '3221': 150, '3231': 89, '3232': 907},
}
CONSULTAS_MANUAL = {'Honduras': 8, 'Ecuador': 22}  # Honduras: 1.1 en dos búsquedas (A y B) + 6 OE
assert sum(BRUTOS_MANUAL['Honduras'].values()) == 1521
assert sum(BRUTOS_MANUAL['Ecuador'].values()) == 13802

for p in PAISES.values():
    s = p['slug']
    p['queries'] = json.load(open(f'../data/queries_{s}.json', encoding='utf-8'))
    p['subcampos'] = pd.read_excel(f'../data/subfields_tagged_{s}.xlsx')
    p['topicos'] = pd.read_excel(f'../outputs/bibliometrics_{s}_db.xlsx', sheet_name='topics')
    p['backup'] = f'../outputs/bibliometrics_{s}_backup'
    p['resumen'] = json.load(open(f"{p['backup']}/run_summary.json", encoding='utf-8'))

In [2]:
MOJIBAKE = re.compile(r'[√¬].|‚Ä.')


def _decodificar(m):
    try:
        return m.group(0).encode('mac_roman').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return m.group(0)


def fix_mojibake(s):
    ''' Corrige secuencias UTF-8 leídas como Mac Roman (p. ej. "√±" -> "ñ"), una a una '''
    return MOJIBAKE.sub(_decodificar, s) if isinstance(s, str) else s


def norm_text(s):
    ''' Minúsculas, sin tildes ni signos, espacios simples '''
    if not isinstance(s, str):
        return ''
    s = unicodedata.normalize('NFKD', fix_mojibake(s))
    s = ''.join(c for c in s if not unicodedata.combining(c)).lower()
    return re.sub(r'[^a-z0-9]+', ' ', s).strip()


def norm_title(s):
    return norm_text(s) or None


def norm_doi(s):
    ''' DOI sin prefijo y en minúsculas; None para vacíos y marcadores ("doi not indicated", "no doi assigned") '''
    if not isinstance(s, str):
        return None
    s = re.sub(r'^(https?://(dx\.)?doi\.org/|doi:\s*)', '', s.strip().lower())
    return s if s.startswith('10.') else None


def norm_oaid(s):
    m = re.search(r'W\d+', s) if isinstance(s, str) else None
    return m.group(0) if m else None


def oe_code(s):
    m = re.match(r'\s*(\d)\.\s*(\d)', str(s))
    return f'{m.group(1)}.{m.group(2)}' if m else None


def oe_de_eje(tid):
    return f'{tid[0]}.{tid[1]}'


def ok(v):
    ''' Valor de llave utilizable (texto no vacío; descarta None y NaN) '''
    return isinstance(v, str) and v != ''


def split_list(v):
    return [x.strip() for x in str(v).split(';') if x.strip()] if pd.notna(v) else []


def patron(terminos):
    ''' Regex de frases exactas (normalizadas) con límites de palabra '''
    terminos = sorted({norm_text(t) for t in terminos} - {''}, key=len, reverse=True)
    return re.compile(r'(?<![a-z0-9])(' + '|'.join(map(re.escape, terminos)) + r')(?![a-z0-9])')


def mapear_ejes(nombres, ejes):
    ''' Nombre de eje (manual) -> thematic_id del pipeline, por similitud del nombre normalizado '''
    destino = {norm_text(e['thematic_axis']): tid for tid, e in ejes.items()}
    out = {}
    for n in nombres:
        c = difflib.get_close_matches(norm_text(n), list(destino), n=1, cutoff=0.6)
        out[n] = destino[c[0]] if c else None
    assert all(out.values()) and len(set(out.values())) == len(out), out
    return out

In [3]:
EJES_HON = list(hon_man.columns[hon_man.columns.get_loc('Pillar ') + 1:hon_man.columns.get_loc('Ranking de pertinencia por OE')])
assert len(EJES_HON) == 20

for nombre, p in PAISES.items():
    q, man, aut = p['queries'], p['man'], p['aut']
    p['ejes'] = {t['thematic_id']: t for t in q['thematics']}
    p['re_kw'] = {tid: patron(e['keywords']) for tid, e in p['ejes'].items()}
    pais = q['meta']['country']
    p['re_pais'] = patron([pais])
    p['re_pais_var'] = patron([t for t in q['meta']['country_terms'] if norm_text(t) != norm_text(pais)])
    p['interes'] = set(p['subcampos'].loc[p['subcampos'].interest_subfield == 1, 'id'])
    p['topico_subcampo'] = dict(zip(p['topicos'].topic_name.map(norm_text), p['topicos'].subfield_id))

    man['_titulo'] = man[p['titulo']].map(norm_title)
    man['_doi'] = man['DOI'].map(norm_doi)
    man['_llave'] = man['_doi'] if nombre == 'Honduras' else man[p['llave']].map(norm_oaid)
    man['_anio'] = pd.to_numeric(man[p['anio']], errors='coerce').astype('Int64')
    man['_oe'] = man[p['oe']].map(oe_code)
    man['_texto'] = (man[p['titulo']].fillna('') + ' ' + man[p['abstract']].fillna('')).map(norm_text)

    aut['_titulo'] = aut['display_name'].map(norm_title)
    aut['_doi'] = aut['doi'].map(norm_doi)
    aut['_llave'] = aut['_doi'] if nombre == 'Honduras' else aut['id'].map(norm_oaid)
    aut['_oe'] = aut['strategic_objective'].map(oe_code)
    aut['_ejes'] = aut['thematic_id'].map(split_list)
    aut['_texto'] = (aut['display_name'].fillna('') + ' ' + aut['abstract'].fillna('')).map(norm_text)

# Eje(s) temático(s) del manual como thematic_id
hm = PAISES['Honduras']['man']
nombres_hon = {c: next(v for v in hm[c].unique() if not v.startswith('Keywords ausentes')) for c in EJES_HON}
mapa_hon = mapear_ejes(nombres_hon.values(), PAISES['Honduras']['ejes'])
hm['_ejes'] = [[mapa_hon[nombres_hon[c]] for c in EJES_HON if fila[c] == nombres_hon[c]] for _, fila in hm[EJES_HON].iterrows()]

em = PAISES['Ecuador']['man']
mapa_ecu = mapear_ejes(em['Eje temático'].unique(), PAISES['Ecuador']['ejes'])
em['_ejes'] = em['Eje temático'].map(lambda e: [mapa_ecu[e]])

print('Ejes del manual mapeados: Honduras', len(mapa_hon), '| Ecuador', len(mapa_ecu))
print('Títulos del manual de Ecuador con mojibake corregido:',
      (em['Título'].map(fix_mojibake) != em['Título']).sum())

Ejes del manual mapeados: Honduras 20 | Ecuador 22
Títulos del manual de Ecuador con mojibake corregido: 796


## 1. Variables compartidas y exclusivas

Las equivalencias entre columnas se definieron a mano. La columna `correspondencia` indica:
- **exacta:** mismo concepto, aunque cambie el formato (prefijos, etiquetas en inglés…).
- **parcial:** concepto parecido con otra definición.
- **derivable:** el dato existe en el pipeline, pero en otra forma.

In [4]:
NOTA_CITAS = 'El manual tiene dos columnas de citas para el mismo concepto'
NOTA_INST = 'En el manual, nombres normalizados a mano; en el pipeline, estandarizados con OpenAlex/ROR'
NOTA_KW = 'El manual trae los concepts de OpenAlex (p. ej. "Government (linguistics)"); el pipeline, sus keywords'
NOTA_REL = 'Posición en el ranking (manual) frente a puntaje de relevancia (pipeline)'
NOTA_COD = 'Coinciden por código'

VARS = {
    'Honduras': [
        ('Título', 'Título', 'display_name', 'exacta', ''),
        ('Resumen', 'Abstract', 'abstract', 'exacta', ''),
        ('Año de publicación', 'Año', 'publication_year', 'exacta', ''),
        ('Autores', 'Autor', 'authorships.author.display_name', 'exacta', ''),
        ('Tópico principal', 'Tópico OpenAlex', 'primary_topic.display_name', 'exacta', ''),
        ('Citas', 'Citation count', 'cited_by_count', 'exacta', NOTA_CITAS),
        ('Citas', 'Cited by', 'cited_by_count', 'exacta', NOTA_CITAS),
        ('FWCI', 'FWCI', 'fwci', 'exacta', ''),
        ('DOI', 'DOI', 'doi', 'exacta', 'Los DOI faltantes del manual se rellenaron con marcadores ("doi not indicated…")'),
        ('Tipo de documento', 'Tipo', 'type', 'exacta', ''),
        ('Instituciones', '3', 'authorships.institutions.display_name', 'exacta', NOTA_INST),
        ('Acceso abierto', 'Open access', 'open_access.is_oa', 'exacta', '"Open Access" / "NOT Open Access" / vacío en el manual'),
        ('Pilar', 'Pillar ', 'pillar', 'exacta', NOTA_COD + '; etiquetas del manual en inglés'),
        ('Objetivo estratégico', 'Objetivo estratégico único (OE)', 'strategic_objective', 'exacta', NOTA_COD + '; etiquetas del manual en inglés'),
        ('Objetivo estratégico', 'Objetivos Estratégicos', 'strategic_objective', 'parcial', 'El manual lista los OE de todas las búsquedas donde apareció el estudio; el pipeline asigna uno'),
        *[('Eje temático', c, 'thematic_axis', 'parcial',
           'Una bandera por eje, calculada con una fórmula sobre título, abstract, keywords y tópico; en el pipeline el eje sale de la consulta')
          for c in EJES_HON],
        ('Palabras clave', 'OpenAlex Keyword', 'keywords', 'parcial', NOTA_KW),
        ('Pertinencia', 'Ranking de pertinencia por OE', 'relevance_score', 'parcial', NOTA_REL),
    ],
    'Ecuador': [
        ('Título', 'Título', 'display_name', 'exacta', ''),
        ('Resumen', 'Abstracto', 'abstract', 'exacta', ''),
        ('Año de publicación', 'Año', 'publication_year', 'exacta', ''),
        ('Autores', 'Autor', 'authorships.author.display_name', 'exacta', ''),
        ('Tópico principal', 'Tópico OpenAlex', 'primary_topic.display_name', 'exacta', ''),
        ('Citas', 'Citation count', 'cited_by_count', 'exacta', NOTA_CITAS),
        ('Citas', 'Cited by', 'cited_by_count', 'exacta', NOTA_CITAS),
        ('FWCI', 'FWCI', 'fwci', 'exacta', ''),
        ('Idioma', 'Idioma', 'language', 'exacta', ''),
        ('DOI', 'DOI', 'doi', 'exacta', 'Sin el prefijo https://doi.org/ en el manual'),
        ('Identificador OpenAlex', 'OpenAlex ID ', 'id', 'exacta', 'Sin el prefijo https://openalex.org/ en el manual'),
        ('Enlace de la publicación', 'Link publicación ', 'primary_location.landing_page_url', 'exacta', ''),
        ('Tipo de documento', 'Tipo', 'type', 'exacta', ''),
        ('Instituciones', 'Institución', 'authorships.institutions.display_name', 'exacta', NOTA_INST),
        ('País de las instituciones', 'País', 'authorships.institutions.country_code', 'exacta', ''),
        ('Pilar / área prioritaria', 'Area prioritaria', 'pillar', 'exacta', NOTA_COD),
        ('Objetivo estratégico', 'Objetivo Estratégico (OE)', 'strategic_objective', 'exacta', NOTA_COD),
        ('Resultado esperado', 'Resultado esperado', 'thematic_id', 'derivable', 'En el pipeline está en la hoja thematics de la base relacional y en los dígitos de thematic_id'),
        ('Eje temático', 'Eje temático', 'thematic_axis', 'exacta', 'En el pipeline un estudio puede tener varios ejes del mismo OE'),
        ('Palabras clave', 'Keyword OpenAlex', 'keywords', 'parcial', NOTA_KW),
        ('Pertinencia', 'Ranking de pertinencia OpenAlex', 'relevance_score', 'parcial', NOTA_REL),
        ('Conceptos de OpenAlex', 'Concepto OpenAlex', None, 'solo manual', 'El pipeline usa topics, que reemplazaron a los concepts en OpenAlex'),
        ('Suma de citas', 'Citations sum', None, 'solo manual', 'Columna vacía'),
        ('Identificador interno', 'ID', None, 'solo manual', 'Número de registro del reporte manual'),
    ],
}

GRUPOS_AUT = [  # (patrón en el nombre de la variable, grupo)
    (r'_imputed$|_source$', 'Trazabilidad de la imputación'),
    (r'^authorships\.institutions', 'Instituciones (id, país, tipo, raíz, grupo de origen)'),
    (r'^authorships', 'Autores'),
    (r'^primary_topic', 'Jerarquía del tópico'),
    (r'open_access|is_oa', 'Acceso abierto'),
    (r'^primary_location|publisher', 'Fuente y editorial'),
    (r'fwci|cited|citation', 'Citas e impacto'),
    (r'thematic|pillar|strategic', 'Estrategia país'),
]


def grupo_aut(c):
    return next((g for pat, g in GRUPOS_AUT if re.search(pat, c)), 'Identificación, fechas y contenido')


def comparar_variables(man, aut, mapeo):
    m = pd.DataFrame(mapeo, columns=['concepto', 'manual', 'automatica', 'correspondencia', 'nota'])
    faltan = (set(m.manual) - set(man.columns)) | (set(m.automatica.dropna()) - set(aut.columns))
    assert not faltan, f'Columnas inexistentes: {faltan}'
    sin_mapear = [c for c in man.columns if c not in set(m.manual) and not c.startswith('_')]
    assert not sin_mapear, f'Columnas del manual sin clasificar: {sin_mapear}'
    compartidas = (m[m.automatica.notna()]
                   .groupby(['concepto', 'automatica', 'correspondencia', 'nota'], sort=False)['manual']
                   .agg(lambda s: ' | '.join(s) if len(s) <= 3 else f'{len(s)} columnas: {" | ".join(s)}')
                   .reset_index()[['concepto', 'manual', 'automatica', 'correspondencia', 'nota']])
    solo_manual = m[m.automatica.isna()][['concepto', 'manual', 'nota']]
    usadas = set(m.automatica.dropna())
    solo_aut = pd.DataFrame({'variable': [c for c in aut.columns if c not in usadas and not c.startswith('_')]})
    solo_aut['grupo'] = solo_aut.variable.map(grupo_aut)
    solo_aut = solo_aut.groupby('grupo', sort=False).variable.agg(['count', ' | '.join]).rename(
        columns={'count': 'n', 'join': 'variables'})
    return compartidas, solo_manual, solo_aut


resumen_vars = {}
for nombre, p in PAISES.items():
    comp, sman, saut = comparar_variables(p['man'], p['aut'], VARS[nombre])
    display(Markdown(f'### {nombre}'))
    display(Markdown(f'**Variables compartidas: {len(comp)} conceptos**'))
    display(comp)
    display(Markdown(f'**Solo en el manual: {len(sman)}**'))
    display(sman if len(sman) else Markdown('_Ninguna: todas las columnas del manual tienen equivalente en el pipeline._'))
    display(Markdown(f'**Solo en el pipeline: {saut.n.sum()} variables**'))
    display(saut)
    resumen_vars[nombre] = {'Columnas manual': p['man'].filter(regex='^[^_]').shape[1],
                            'Columnas automática': p['aut'].filter(regex='^[^_]').shape[1],
                            'Conceptos compartidos': len(comp),
                            '  exactos': (comp.correspondencia == 'exacta').sum(),
                            '  parciales o derivables': (comp.correspondencia != 'exacta').sum(),
                            'Solo manual': len(sman), 'Solo automática': saut.n.sum()}
display(Markdown('### Resumen'))
display(pd.DataFrame(resumen_vars))

### Honduras

**Variables compartidas: 17 conceptos**

,concepto,manual,automatica,correspondencia,nota
0,Título,Título,display_name,exacta,
1,Resumen,Abstract,abstract,exacta,
2,Año de publicación,Año,publication_year,exacta,
3,Autores,Autor,authorships.author.display_name,exacta,
4,Tópico principal,Tópico OpenAlex,primary_topic.display_name,exacta,
5,Citas,Citation count | Cited by,cited_by_count,exacta,El manual tiene dos columnas de citas para el mismo concepto
6,FWCI,FWCI,fwci,exacta,
7,DOI,DOI,doi,exacta,"Los DOI faltantes del manual se rellenaron con marcadores (""doi not indicated…"")"
8,Tipo de documento,Tipo,type,exacta,
9,Instituciones,3,authorships.institutions.display_name,exacta,"En el manual, nombres normalizados a mano; en el pipeline, estandarizados con OpenAlex/ROR"


**Solo en el manual: 0**

_Ninguna: todas las columnas del manual tienen equivalente en el pipeline._

**Solo en el pipeline: 32 variables**

... (6 outputs left out)

**Solo en el pipeline: 28 variables**

,n,variables
grupo,,
"Identificación, fechas y contenido",4,publication_date | openalex_updated_date | sustainable_development_goals.display_name | retrieved_date
Fuente y editorial,2,primary_location.raw_source_name | publisher
Acceso abierto,4,primary_location.source.is_oa | open_access.is_oa | open_access.oa_status | open_access.oa_url
Citas e impacto,1,citation_normalized_percentile.value
Jerarquía del tópico,7,primary_topic.id | primary_topic.subfield.id | primary_topic.subfield.display_name | primary_topic.field.i...
"Instituciones (id, país, tipo, raíz, grupo de origen)",4,authorships.institutions.id | authorships.institutions.type | authorships.institutions.root_display_name |...
Trazabilidad de la imputación,6,abstract_source | language_imputed | source_is_oa_imputed | publisher_source | authorships.author.id_imput...


### Resumen

,Honduras,Ecuador
Columnas manual,37,24
Columnas automática,48,48
Conceptos compartidos,17,20
exactos,13,17
parciales o derivables,4,3
Solo manual,0,3
Solo automática,32,28


## 2. Estudios compartidos y exclusivos

### 2.1 Duplicados internos por tipo de estudio
Antes del cruce se revisa cuántos registros repetidos trae cada base. Columnas:
- `filas_en_duplicados`: filas cuya llave aparece más de una vez.
- `grupos_duplicados`: llaves repetidas distintas.
- `filas_sobrantes`: filas que sobrarían si se dejara una por grupo.

Un grupo puede incluir tipos distintos, así que la suma de grupos por tipo puede superar el total.

Los duplicados por título se cuentan solo entre títulos de 4 palabras o más, igual que en el pipeline, para no fusionar títulos genéricos como "Editorial" o "Ecuador". Los identificadores (DOI u OpenAlex ID) se comparan sin esa restricción.

In [5]:
MIN_PALABRAS = 4  # igual que el pipeline: no se fusionan títulos genéricos ("Editorial", "Ecuador")


def titulo_largo(t):
    return t if ok(t) and len(t.split()) >= MIN_PALABRAS else None


def duplicados_por_tipo(llave, tipo):
    d = pd.DataFrame({'llave': llave.values, 'tipo': tipo.fillna('sin dato').values})
    d['dup'] = d.llave.notna() & d.llave.duplicated(keep=False)
    out = pd.DataFrame({'filas': d.groupby('tipo').size(),
                        'filas_en_duplicados': d.groupby('tipo').dup.sum(),
                        'grupos_duplicados': d[d.dup].groupby('tipo').llave.nunique()}).fillna(0).astype(int)
    out['filas_sobrantes'] = d[d.dup].groupby('tipo').llave.apply(lambda s: len(s) - s.nunique()).reindex(out.index).fillna(0).astype(int)
    grupos = d.loc[d.dup, 'llave'].nunique()
    out.loc['Total'] = [len(d), d.dup.sum(), grupos, d.dup.sum() - grupos]
    return out.sort_values('filas', ascending=False)


duplicados = {}
for nombre, p in PAISES.items():
    man, aut = p['man'], p['aut']
    tablas = {'Manual · título': duplicados_por_tipo(man._titulo.map(titulo_largo), man[p['tipo']]),
              f"Manual · {p['nombre_llave']}": duplicados_por_tipo(man._llave, man[p['tipo']]),
              'Automática · título': duplicados_por_tipo(aut._titulo.map(titulo_largo), aut.type)}
    duplicados[nombre] = tablas
    display(Markdown(f'#### {nombre}'))
    display(pd.concat(tablas, axis=1).fillna(0).astype(int))

#### Honduras

Manual · título                                                       Manual · doi                                                       Automática · título                                                      
                           filas filas_en_duplicados grupos_duplicados filas_sobrantes        filas filas_en_duplicados grupos_duplicados filas_sobrantes               filas filas_en_duplicados grupos_duplicados filas_sobrantes
tipo                                                                                                                                                                                                                               
Total                       1187                  93                39              54         1187                   0                 0               0                1047                   0                 0               0
article                      949                  56                27              29          949                   0                 0               0                 729                   0                 0               0
book-chapter                  67                   0                 0               0           67                   0                 0               0                  61                   0                 0               0
book                          49                   0                 0               0           49                   0                 0               0                  46                   0                 0               0
dataset                       48                  33                11              22           48                   0                 0               0                  16                   0                 0               0
dissertation                  37                   4                 2               2           37                   0                 0               0                  60                   0                 0               0
report                        37                   0                 0               0           37                   0                 0               0                  73                   0                 0               0
conference-paper               0                   0                 0               0            0                   0                 0               0                  62                   0                 0               0

#### Ecuador

Manual · título                                                       Manual · openalex_id                                                       Automática · título                                                      
                           filas filas_en_duplicados grupos_duplicados filas_sobrantes                filas filas_en_duplicados grupos_duplicados filas_sobrantes               filas filas_en_duplicados grupos_duplicados filas_sobrantes
tipo                                                                                                                                                                                                                                       
Total                      11595                 907               445             462                11595                 624               307             317               16014                   2                 1               1
article                     8885                 723               375             348                 8885                 509               252             257               11138                   0                 0               0
dissertation                1397                 125                63              62                 1397                 113                55              58                3127                   2                 1               1
conference-paper             462                  15                13               2                  462                   0                 0               0                 676                   0                 0               0
book-chapter                 357                   9                 7               2                  357                   0                 0               0                 480                   0                 0               0
book                         305                   7                 6               1                  305                   0                 0               0                 384                   0                 0               0
report                       115                   8                 8               0                  115                   0                 0               0                 141                   0                 0               0
dataset                       71                  19                14               5                   71                   2                 1               1                  64                   0                 0               0
data-paper                     3                   1                 1               0                    3                   0                 0               0                   4                   0                 0               0

### 2.2 Cruce
- **Honduras:** se cruza por DOI normalizado. Las filas sin DOI válido (marcadores "doi not indicated…" o "no doi assigned…") se cruzan por título normalizado.
- **Ecuador:** se cruza por OpenAlex ID. Las filas sin ID se cruzan por título normalizado.

El título se compara contra todos los títulos de la base automática. Si coincide con varios estudios, se prefiere el del mismo año, y el caso queda marcado como ambiguo.

**Sensibilidad.** Las filas que tienen llave pero no cruzan se intentan también por título. El resultado se informa aparte y no cambia la regla principal.

In [ ]:
def indice_titulos(aut):
    idx = {}
    for i, t, y in zip(aut['id'], aut['_titulo'], aut['publication_year']):
        if ok(t):
            idx.setdefault(t, []).append((i, y))
    return idx


def buscar_titulo(t, y, idx):
    cands = idx.get(t) if ok(t) else None
    if not cands:
        return None, False
    mismo_anio = [i for i, a in cands if a == y]
    return (mismo_anio or [i for i, _ in cands])[0], len(cands) > 1


def cruzar(man, aut, nombre_llave):
    por_llave = {k: i for k, i in zip(aut['_llave'], aut['id']) if ok(k)}
    idx = indice_titulos(aut)
    filas = []
    for k, t, y in zip(man['_llave'], man['_titulo'], man['_anio']):
        if ok(k):
            aid = por_llave.get(k)
            sens = None if aid else buscar_titulo(t, y, idx)[0]
            filas.append((nombre_llave, aid, nombre_llave if aid else None, False, sens))
        else:
            aid, amb = buscar_titulo(t, y, idx)
            filas.append(('título', aid, 'título' if aid else None, amb and aid is not None, None))
    r = pd.DataFrame(filas, index=man.index, columns=['regla', 'aut_id', 'metodo', 'ambiguo', 'aut_id_sensibilidad'])
    return man.join(r)


def resumen_cruce(m, aut, nombre_llave):
    ids = set(m.aut_id.dropna())
    assert ids <= set(aut['id'])
    sens = set(m.aut_id_sensibilidad.dropna()) - ids
    solo_man = m[m.aut_id.isna()]
    return pd.Series({
        'Llave principal': nombre_llave,
        'Filas en el manual': len(m),
        '  con llave válida': (m.regla == nombre_llave).sum(),
        '  sin llave (se cruzan por título)': (m.regla == 'título').sum(),
        'Filas manuales cruzadas por llave': (m.metodo == nombre_llave).sum(),
        'Filas manuales cruzadas por título': (m.metodo == 'título').sum(),
        '  de ellas con título ambiguo': int(m.ambiguo.sum()),
        'Filas manuales que repiten un estudio ya cruzado': m.aut_id.notna().sum() - len(ids),
        'Filas solo en el manual': len(solo_man),
        '  títulos únicos': solo_man._titulo.nunique(),
        'Estudios en la base automática': len(aut),
        'Estudios en ambas bases': len(ids),
        'Estudios solo en la base automática': len(aut) - len(ids),
        'Sensibilidad: filas con llave sin cruce que cruzan por título': m.aut_id_sensibilidad.notna().sum(),
    })


resumen = {}
for nombre, p in PAISES.items():
    m = cruzar(p['man'], p['aut'], p['nombre_llave'])
    assert m.regla.notna().all() and len(m) == len(p['man'])
    p['cruce'] = m
    p['aut']['en_manual'] = p['aut']['id'].isin(set(m.aut_id.dropna()))
    resumen[nombre] = resumen_cruce(m, p['aut'], p['nombre_llave'])
    r = resumen[nombre]
    assert r['Estudios en ambas bases'] + r['Estudios solo en la base automática'] == len(p['aut'])
    assert r['Filas solo en el manual'] + r['Filas manuales cruzadas por llave'] + r['Filas manuales cruzadas por título'] == len(p['man'])
display(pd.concat(resumen, axis=1))

,Honduras,Ecuador
Llave principal,doi,openalex_id
Filas en el manual,1187,11595
con llave válida,1058,11432
sin llave (se cruzan por título),129,163
Filas manuales cruzadas por llave,596,9394
Filas manuales cruzadas por título,61,73
de ellas con título ambiguo,0,8
Filas manuales que repiten un estudio ya cruzado,9,246
Filas solo en el manual,530,2128
títulos únicos,495,2012


Revisión de 5 cruces por título elegidos al azar (título manual frente al automático):

In [7]:
for nombre, p in PAISES.items():
    m = p['cruce']
    muestra = m[m.metodo == 'título'].sample(min(5, (m.metodo == 'título').sum()), random_state=1)
    muestra = muestra.join(p['aut'].set_index('id')[['display_name', 'publication_year']], on='aut_id')
    display(Markdown(f'**{nombre}**'))
    display(muestra[[p['titulo'], 'display_name', '_anio', 'publication_year', 'ambiguo']])

**Honduras**

,Título,display_name,_anio,publication_year,ambiguo
996,Ancestral Resistance: Women and LGBTTI+ Lenca Activism Against Extractivist Projects,Ancestral Resistance: Women and LGBTTI+ Lenca Activism Against Extractivist Projects,2026,2026,False
1019,Análisis comparativo de intervenciones en seguridad y salud en el trabajo (SST) orientado a las poblacione...,Análisis comparativo de intervenciones en seguridad y salud en el trabajo (SST) orientado a las poblacione...,2025,2025,False
58,Análisis de las Relaciones Comerciales entre países del Triángulo Norte Centroamericano (República de Guat...,Análisis de las Relaciones Comerciales entre países del Triángulo Norte Centroamericano (República de Guat...,2019,2019,False
1058,Fundraising for Families: A strategic communications campaign for Give Hope 2 Kids in Honduras,Fundraising for Families: A strategic communications campaign for Give Hope 2 Kids in Honduras,2020,2020,False
1077,"Nuestra Tierra: Land, Identity, and the Ch’orti’ Maya Indigenous Rights Movements","Nuestra Tierra: Land, Identity, and the Ch’orti’ Maya Indigenous Rights Movements",2025,2025,False


**Ecuador**

,Título,display_name,_anio,publication_year,ambiguo
3161,Potencial de la biomasa y la cinética del modelo de secado de Piptocoma discolor (pig√ºe) como fuente de e...,Potencial de la biomasa y la cinética del modelo de secado de Piptocoma discolor (pigüe) como fuente de en...,2020,2021,False
8773,Comparació tècnica i econ√≤mica entre un sistema de fangs activats i aiguamolls artificials per a la depur...,Comparació tècnica i econòmica entre un sistema de fangs activats i aiguamolls artificials per a la depura...,2026,2026,False
579,A possible mechanism for partial crowding‚Äêout of R&D subsidies in developing countries,A possible mechanism for partial crowding‐out of R&D subsidies in developing countries,2023,2023,False
11187,Anàlisi comparada de les normatives de ciberseguretat: Unió Europea vs Equador. Propostes te√≤riques per a...,Anàlisi comparada de les normatives de ciberseguretat: Unió Europea vs Equador. Propostes teòriques per a ...,2026,2026,False
8750,Anàlisi i propostes d'optimització hídrica i energètica del sistema de prove√Øment d'aigua potable del can...,Anàlisi i propostes d'optimització hídrica i energètica del sistema de proveïment d'aigua potable del cant...,2025,2025,False


Coincidencias de la prueba de sensibilidad (filas con llave sin cruce que sí coinciden por título):

In [8]:
for nombre, p in PAISES.items():
    m = p['cruce']
    s = m[m.aut_id_sensibilidad.notna()].join(p['aut'].set_index('id')[['display_name', 'doi']], on='aut_id_sensibilidad')
    display(Markdown(f"**{nombre}**: {len(s)} filas"))
    display(s[[p['titulo'], p['llave'], 'display_name', 'doi']].head(10))

**Honduras**: 26 filas

,Título,DOI,display_name,doi
80,Honduras,https://doi.org/10.5089/9798400255496.002,Honduras,https://doi.org/10.1787/5faef716-en
103,Gestión del riesgo de desastres en la formación docente de Universidad Pedagógica de Honduras,https://doi.org/10.5281/zenodo.18086547,Gestión del riesgo de desastres en la formación docente de Universidad Pedagógica de Honduras,https://doi.org/10.5281/zenodo.18086546
162,"Gangs and Modern-Day Slavery in El Salvador, Honduras and Guatemala: A Non-Traditional Model of Human Traf...",https://doi.org/10.1080/23322705.2020.1719343,"Gangs and Modern-Day Slavery in El Salvador, Honduras and Guatemala: A Non-Traditional Model of Human Traf...",https://doi.org/10.2139/ssrn.3539953
251,Gasto Público en Salud Mental en Iberoamérica (2000-2026): Base de Datos por País y Categoría,https://doi.org/10.17632/dmyc5xtn62.2,Gasto Público en Salud Mental en Iberoamérica (2000-2026): Base de Datos por País y Categoría,https://doi.org/10.17632/dmyc5xtn62
256,Gasto Público en Salud Mental en Iberoamérica (2000-2026): Base de Datos por País y Categoría,https://doi.org/10.17632/dmyc5xtn62.1,Gasto Público en Salud Mental en Iberoamérica (2000-2026): Base de Datos por País y Categoría,https://doi.org/10.17632/dmyc5xtn62
287,VOLUMEN TRES MILIMETRO,https://doi.org/10.5281/zenodo.19324058,VOLUMEN TRES MILIMETRO,https://doi.org/10.5281/zenodo.19324057
296,Experimental Long-Term Effects of Early-Childhood and School-Age Exposure to a Conditional Cash Transfer P...,https://doi.org/10.2139/ssrn.3449584,Experimental long-term effects of early-childhood and school-age exposure to a conditional cash transfer p...,https://doi.org/10.1016/j.jdeveco.2019.102385
348,"Delitos contra Menores en América Latina: Estadísticas por País, Tipo de Delito y Rango de Edad (2020-2024)",https://doi.org/10.17632/6nnxg7kmb9.1,"Delitos contra Menores en América Latina: Estadísticas por País, Tipo de Delito y Rango de Edad (2020-2024)",https://doi.org/10.17632/6nnxg7kmb9
351,"Delitos contra Menores en América Latina: Estadísticas por País, Tipo de Delito y Rango de Edad (2020-2024)",https://doi.org/10.17632/6nnxg7kmb9.2,"Delitos contra Menores en América Latina: Estadísticas por País, Tipo de Delito y Rango de Edad (2020-2024)",https://doi.org/10.17632/6nnxg7kmb9
469,Empleo en Latinoamérica: Base de Datos por País con Evolución 2000-2020 (cada 5 años),https://doi.org/10.6084/m9.figshare.31441840.v1,Empleo en Latinoamérica: Base de Datos por País con Evolución 2000-2020 (cada 5 años),https://doi.org/10.6084/m9.figshare.31441840


**Ecuador**: 327 filas

,Título,OpenAlex ID,display_name,doi
113,The 2023 Latin America report of the Lancet Countdown on health and climate change: the imperative for hea...,W4395026061,The 2023 Latin America report of the Lancet Countdown on health and climate change: the imperative for hea...,NaN
418,Does The Distribution of Income Respond to the Development of Institutions?,W4413919538,Does The Distribution of Income Respond to the Development of Institutions?,https://doi.org/10.33899/tanra.2022.176210.1185
427,An anatomy of external shocks in the Andean region,W4323051472,An Anatomy of External Shocks in the Andean Region,https://doi.org/10.18235/0001949
437,Financing sustainable development: the role of institutional quality in Latin America,W7104882205,Financing sustainable development: the role of institutional quality in Latin America,https://doi.org/10.1504/ijge.2025.149741
520,La gestión de los recursos públicos como factor determinante del desarrollo social: una mirada desde la ef...,W7130408242,La gestión de los recursos públicos como factor determinante del desarrollo social: una mirada desde la ef...,https://doi.org/10.65415/rcs.v4i1.27
559,"Replication Data and Code for: ""Homicidal Violence and State Capacity in Latin America: The Story of the P...",W7134122897,"Replication Data and Code for: ""Homicidal Violence and State Capacity in Latin America: The Story of the P...",https://doi.org/10.5281/zenodo.18902585
624,Digital teaching competence of higher education professors: self-perception study in an Ecuadorian university,W4403866144,Digital teaching competence of higher education professors: self-perception study in an Ecuadorian university,https://doi.org/10.12688/f1000research.139064.2
644,Estudio de la Brecha Digital y el Proceso de Enseñanza-Aprendizaje en Ecuador - Caso De Estudio: Universid...,W4388179100,Estudio de la Brecha Digital y el Proceso de Enseñanza- Aprendizaje en Ecuador - Caso De Estudio: Universi...,https://doi.org/10.54580/r0402.06
688,Prevalence of gender-based violence in university nursing students: Multi-centric Mexican study,W4311259884,Prevalence of gender-based violence in university nursing students: Multi-centric Mexican study,https://doi.org/10.30574/wjarr.2022.16.03.1215
837,The Labor Market Effects of Venezuelan Migration in Ecuador,W3043270856,The Labor Market Effects of Venezuelan Migration in Ecuador,https://doi.org/10.2139/ssrn.3654938


### 2.3 Perfil de los grupos
Distribución de los tres grupos (ambas bases, solo manual, solo automático) por año, tipo y objetivo estratégico
- Para los estudios presentes en ambas bases se usan los valores del pipeline.
- Los porcentajes se calculan dentro de cada grupo.

In [9]:
ORDEN = ['Ambas', 'Solo manual', 'Solo automático', 'Total']


def tabla_grupos(p):
    m, aut = p['cruce'], p['aut']
    a = pd.DataFrame({'grupo': aut.en_manual.map({True: 'Ambas', False: 'Solo automático'}),
                      'Año': aut.publication_year, 'Tipo': aut.type, 'OE': aut._oe, 'Idioma': aut.language})
    sm = m[m.aut_id.isna()]
    b = pd.DataFrame({'grupo': 'Solo manual', 'Año': sm._anio, 'Tipo': sm[p['tipo']], 'OE': sm._oe,
                      'Idioma': sm[p['idioma']] if p['idioma'] else None})
    return pd.concat([a, b], ignore_index=True)


def perfil(g, var, top=None):
    v = g[var].astype('string').fillna('sin dato')
    if top:
        principales = v.value_counts().index[:top]
        v = v.where(v.isin(principales), 'otros')
    t = pd.crosstab(v, g.grupo, margins=True, margins_name='Total').reindex(columns=ORDEN, fill_value=0)
    pct = (t.div(t.loc['Total']) * 100).round(1)
    return pd.concat({'n': t, '%': pct}, axis=1)


for nombre, p in PAISES.items():
    g = tabla_grupos(p)
    p['grupos'] = g
    display(Markdown(f'#### {nombre}'))
    for var, top in [('Año', None), ('Tipo', None), ('OE', None)]:
        display(perfil(g, var, top))

#### Honduras

n                                        %                                   
grupo Ambas Solo manual Solo automático Total  Ambas Solo manual Solo automático  Total
Año                                                                                    
2019     71          65              26   162   11.0        12.3             6.5   10.3
2020     92          56              30   178   14.2        10.6             7.5   11.3
2021     90          81              31   202   13.9        15.3             7.8   12.8
2022     82          36              40   158   12.7         6.8            10.0   10.0
2023     70          60              42   172   10.8        11.3            10.5   10.9
2024     91          65              43   199   14.0        12.3            10.8   12.6
2025    100          98              60   258   15.4        18.5            15.0   16.4
2026     52          69             127   248    8.0        13.0            31.8   15.7
Total   648         530             399  1577  100.0       100.0           100.0  100.0

n                                        %                                   
grupo            Ambas Solo manual Solo automático Total  Ambas Solo manual Solo automático  Total
Tipo                                                                                              
article            476         420             253  1149   73.5        79.2            63.4   72.9
book                24          23              22    69    3.7         4.3             5.5    4.4
book-chapter        41          26              20    87    6.3         4.9             5.0    5.5
conference-paper    36           0              26    62    5.6         0.0             6.5    3.9
dataset             11          37               5    53    1.7         7.0             1.3    3.4
dissertation        24          18              36    78    3.7         3.4             9.0    4.9
report              36           6              37    79    5.6         1.1             9.3    5.0
Total              648         530             399  1577  100.0       100.0           100.0  100.0

n                                        %                                   
grupo Ambas Solo manual Solo automático Total  Ambas Solo manual Solo automático  Total
OE                                                                                     
1.1      62          23              22   107    9.6         4.3             5.5    6.8
1.2     311         149             179   639   48.0        28.1            44.9   40.5
2.1      30           7              27    64    4.6         1.3             6.8    4.1
2.2      34         275              19   328    5.2        51.9             4.8   20.8
3.1      31           4               9    44    4.8         0.8             2.3    2.8
3.2      32          20              31    83    4.9         3.8             7.8    5.3
3.3     148          52             112   312   22.8         9.8            28.1   19.8
Total   648         530             399  1577  100.0       100.0           100.0  100.0

#### Ecuador

n                                         %                                   
grupo Ambas Solo manual Solo automático  Total  Ambas Solo manual Solo automático  Total
Año                                                                                     
2019    807         211             656   1674    8.8         9.9             9.7    9.2
2020    703         203             473   1379    7.6         9.5             7.0    7.6
2021    704         162             411   1277    7.6         7.6             6.1    7.0
2022    628         179             416   1223    6.8         8.4             6.1    6.7
2023    849         177             587   1613    9.2         8.3             8.6    8.9
2024   1478         244             854   2576   16.0        11.5            12.6   14.2
2025   2367         508            1387   4262   25.7        23.9            20.4   23.5
2026   1685         443            2009   4137   18.3        20.8            29.6   22.8
2027      0           1               0      1    0.0         0.0             0.0    0.0
Total  9221        2128            6793  18142  100.0       100.0           100.0  100.0

n                                         %                                   
grupo            Ambas Solo manual Solo automático  Total  Ambas Solo manual Solo automático  Total
Tipo                                                                                               
article           7002        1671            4136  12809   75.9        78.5            60.9   70.6
book               260          45             124    429    2.8         2.1             1.8    2.4
book-chapter       296          73             184    553    3.2         3.4             2.7    3.0
conference-paper   397          66             279    742    4.3         3.1             4.1    4.1
data-paper           3           0               1      4    0.0         0.0             0.0    0.0
dataset             26          45              38    109    0.3         2.1             0.6    0.6
dissertation      1145         202            1982   3329   12.4         9.5            29.2   18.3
report              92          26              49    167    1.0         1.2             0.7    0.9
Total             9221        2128            6793  18142  100.0       100.0           100.0  100.0

n                                         %                                   
grupo Ambas Solo manual Solo automático  Total  Ambas Solo manual Solo automático  Total
OE                                                                                      
1.1    4297        1014            3777   9088   46.6        47.7            55.6   50.1
1.2     815          97             398   1310    8.8         4.6             5.9    7.2
2.1    1907         667            1103   3677   20.7        31.3            16.2   20.3
3.1    1283         186             901   2370   13.9         8.7            13.3   13.1
3.2     919         164             614   1697   10.0         7.7             9.0    9.4
Total  9221        2128            6793  18142  100.0       100.0           100.0  100.0

### 2.4 Concordancia en los estudios que coinciden
Para cada fila del manual que cruzó se compara:
- el año;
- el tipo;
- el objetivo estratégico, por código;
- el eje temático: si alguno de los ejes del manual coincide con alguno del pipeline;
- las citas. Aquí se esperan diferencias, porque las fechas de extracción son distintas.

In [10]:
def concordancia(p):
    m = p['cruce'].dropna(subset=['aut_id'])
    j = m.join(p['aut'].set_index('id'), on='aut_id', rsuffix='_aut')
    citas_man = pd.to_numeric(j[p['citas']], errors='coerce')
    dif = j.cited_by_count - citas_man
    eje_comun = [bool(set(a) & set(b)) for a, b in zip(j['_ejes'], j['_ejes_aut'])]
    con_eje = j['_ejes'].str.len() > 0
    return pd.Series({
        'Filas cruzadas': len(j),
        'Mismo año (%)': (j._anio == j.publication_year).mean() * 100,
        'Mismo tipo (%)': (j[p['tipo']] == j.type).mean() * 100,
        'Mismo OE (%)': (j._oe == j._oe_aut).mean() * 100,
        'Filas del manual con eje asignado': int(con_eje.sum()),
        'Algún eje en común (% de las que tienen eje)': pd.Series(eje_comun)[con_eje.values].mean() * 100,
        'Citas: mediana (pipeline − manual)': dif.median(),
        'Citas: pipeline ≥ manual (%)': (dif >= 0).mean() * 100,
    }).round(1), pd.crosstab(j._oe.fillna('sin dato'), j._oe_aut.fillna('sin dato'),
                             rownames=['OE manual'], colnames=['OE pipeline'], margins=True, margins_name='Total')


conc = {}
for nombre, p in PAISES.items():
    conc[nombre], oe_tab = concordancia(p)
    display(Markdown(f'**{nombre}: OE del manual (filas) frente a OE del pipeline (columnas)**'))
    display(oe_tab)
display(pd.DataFrame(conc))

**Honduras: OE del manual (filas) frente a OE del pipeline (columnas)**

OE pipeline,1.1,1.2,2.1,2.2,3.1,3.2,3.3,Total
OE manual,,,,,,,,
1.1,59,2,0,0,0,0,2,63
1.2,0,289,0,1,0,0,2,292
2.1,1,0,30,0,0,1,3,35
2.2,1,19,0,33,1,3,6,63
3.1,0,0,0,0,31,0,0,31
3.2,0,2,0,0,0,30,2,34
3.3,1,3,0,0,0,0,135,139
Total,62,315,30,34,32,34,150,657


**Ecuador: OE del manual (filas) frente a OE del pipeline (columnas)**

OE pipeline,1.1,1.2,2.1,3.1,3.2,Total
OE manual,,,,,,
1.1,4260,23,69,23,5,4380
1.2,19,795,6,0,0,820
2.1,37,11,1852,1,0,1901
3.1,79,2,17,1263,11,1372
3.2,28,2,3,37,924,994
Total,4423,833,1947,1324,940,9467


,Honduras,Ecuador
Filas cruzadas,657.0,9467.0
Mismo año (%),98.6,99.3
Mismo tipo (%),91.9,99.7
Mismo OE (%),92.4,96.1
Filas del manual con eje asignado,503.0,9467.0
Algún eje en común (% de las que tienen eje),96.0,96.0
Citas: mediana (pipeline − manual),1.0,1.0
Citas: pipeline ≥ manual (%),97.6,99.2


## 3. Causas de las diferencias

### 3.1 Embudo comparado
- En los manuales los brutos son la suma de los resultados de las consultas que informa cada PDF.
- En el pipeline los brutos son la suma de los registros únicos de cada consulta (`thematic_*.json`). Un estudio que aparece en varios ejes cuenta varias veces.
- Los títulos repetidos de la base final se toman de 2.1: se cuentan solo entre títulos de 4 palabras o más, igual que en el pipeline, para no fusionar títulos genéricos como "Editorial" o "Ecuador". Pueden quedar repetidos que el pipeline no fusiona porque difieren solo en tildes o puntuación, que la normalización del notebook sí elimina.

Se cargan los respaldos crudos, que también se usan en 3.3 y 3.4.

In [11]:
def cargar_crudos(carpeta):
    filas, brutos = [], {}
    for f in sorted(glob.glob(f'{carpeta}/thematic_*.json')):
        tid = re.search(r'thematic_(\d+)\.json$', f).group(1)
        with open(f, encoding='utf-8') as fh:
            d = json.load(fh)
        vistos = set()
        for w in d['results']:
            if w['id'] in vistos:
                continue
            vistos.add(w['id'])
            sf = ((w.get('primary_topic') or {}).get('subfield') or {}).get('id')
            filas.append((tid, w['id'], w.get('doi'), w.get('display_name'),
                          int(sf.rsplit('/', 1)[-1]) if sf else None, (w.get('created_date') or '')[:10]))
        brutos[tid] = len(vistos)
        del d
    c = pd.DataFrame(filas, columns=['thematic_id', 'id', 'doi', 'titulo', 'subfield_id', 'created_date'])
    obras = c.groupby('id').agg(doi=('doi', 'first'), titulo=('titulo', 'first'), subfield_id=('subfield_id', 'first'),
                                created_date=('created_date', 'first'), ejes=('thematic_id', lambda s: sorted(set(s))))
    return obras, brutos


for nombre, p in PAISES.items():
    obras, brutos = cargar_crudos(p['backup'])
    assert sum(brutos.values()) == p['resumen']['stages']['queries']['studies']
    obras['_doi'] = obras.doi.map(norm_doi)
    obras['_titulo'] = obras.titulo.map(norm_title)
    obras['interes'] = obras.subfield_id.isin(p['interes'])
    obras['en_aut'] = obras.index.isin(set(p['aut']['id']))
    assert obras.en_aut.sum() == len(p['aut'])
    p['obras'], p['brutos_aut'] = obras, brutos
    print(nombre, '| registros crudos:', sum(brutos.values()), '| obras únicas:', len(obras))

Honduras | registros crudos: 1442 | obras únicas: 1291


Ecuador | registros crudos: 22288 | obras únicas: 19487


Recordar que el flujo/pipeline primero se filtra por subcampo y luego duplicados, en la celda anterior solo se hace la identificación de duplicados para obras únicas.

In [12]:
embudo = {}
for nombre, p in PAISES.items():
    st = p['resumen']['stages']
    dup_man = duplicados[nombre]['Manual · título'].loc['Total', 'filas_sobrantes']
    dup_aut = duplicados[nombre]['Automática · título'].loc['Total', 'filas_sobrantes']
    embudo[(nombre, 'Manual')] = {
        'Consultas': CONSULTAS_MANUAL[nombre],
        'Términos de país': 'solo el nombre del país',
        'Palabras clave': 'frase exacta',
        'Resultados brutos': sum(BRUTOS_MANUAL[nombre].values()),
        'Obras únicas en los resultados brutos': 'no informado',
        'Filtro temático': 'tópicos eliminados a mano y revisión',
        'Tras el filtro temático': 'no informado',
        'Base final': len(p['man']),
        f'Títulos repetidos ({MIN_PALABRAS}+ palabras) que quedan en la base final': dup_man,
        'Fecha de extracción (a más tardar)': p['fecha_manual'],
    }
    embudo[(nombre, 'Automático')] = {
        'Consultas': st['queries']['thematics'],
        'Términos de país': f"{len(p['queries']['meta']['country_terms'])} términos",
        'Palabras clave': 'stemming' if p['resumen']['search']['stemming'] else 'frase exacta',
        'Resultados brutos': st['queries']['studies'],
        'Obras únicas en los resultados brutos': len(p['obras']),
        'Filtro temático': 'subcampos de interés',
        'Tras el filtro temático': st['subfields']['studies'],
        'Base final': st['duplicates']['studies'],
        f'Títulos repetidos ({MIN_PALABRAS}+ palabras) que quedan en la base final': dup_aut,
        'Fecha de extracción (a más tardar)': p['resumen']['start'][:10],
    }
display(pd.DataFrame(embudo).astype(str))

Honduras                                                     Ecuador                      
                                                                                           Manual            Automático                                Manual            Automático
Consultas                                                                                       8                    20                                    22                    22
Términos de país                                                          solo el nombre del país           11 términos               solo el nombre del país           11 términos
Palabras clave                                                                       frase exacta              stemming                          frase exacta              stemming
Resultados brutos                                                                            1521                  1442                                 13802                 22288
Obras únicas en los resultados brutos                                                no informado                  1291                          no informado                 19487
Filtro temático                                              tópicos eliminados a mano y revisión  subcampos de interés  tópicos eliminados a mano y revisión  subcampos de interés
Tras el filtro temático                                                              no informado                  1192                          no informado                 18749
Base final                                                                                   1187                  1047                                 11595                 16014
Títulos repetidos (4+ palabras) que quedan en la base final                                    54                     0                                   462                     1
Fecha de extracción (a más tardar)                                                     2026-07-01            2026-11-20                            2026-10-24            2026-11-20

**Queda 1 fila repetida:** "T-EC 0023/CD 14941" (dos tesis de 2025). En el título original tiene 3 palabras, así que la regla de 4 palabras lo protege como título genérico.


### 3.2 Comparación por eje (Ecuador) y por objetivo estratégico (Honduras)
- **Ecuador:** las dos aproximaciones hicieron una consulta por eje, así que los brutos se pueden comparar eje por eje.
- **Honduras:** el manual hizo las consultas por OE. Los brutos del pipeline se suman por OE, de modo que un estudio que aparece en dos ejes del mismo OE cuenta dos veces.

En las bases finales se usan los ejes de cada fila: para el manual, el eje asignado (Ecuador) o las banderas (Honduras); para el pipeline, `thematic_id`.

In [13]:
p = PAISES['Ecuador']
t = pd.DataFrame({'Eje temático': {tid: e['thematic_axis'] for tid, e in p['ejes'].items()},
                  'Keywords pipeline': {tid: len(e['keywords']) for tid, e in p['ejes'].items()}})
t['Brutos manual (PDF)'] = pd.Series(BRUTOS_MANUAL['Ecuador'])
t['Brutos pipeline'] = pd.Series(p['brutos_aut'])
t['Razón brutos'] = (t['Brutos pipeline'] / t['Brutos manual (PDF)']).round(2)
t['Final manual'] = p['man']._ejes.explode().value_counts()
t['Final pipeline'] = p['aut']._ejes.explode().value_counts()
t['Razón final'] = (t['Final pipeline'] / t['Final manual']).round(2)
t = t.sort_values('Razón brutos', ascending=False)
total = t.drop(columns=['Eje temático']).sum()
total['Razón brutos'] = round(total['Brutos pipeline'] / total['Brutos manual (PDF)'], 2)
total['Razón final'] = round(total['Final pipeline'] / total['Final manual'], 2)
t.loc['Total'] = ['Total', *total[t.columns[1:]]]
eje_ecu = t
display(Markdown('#### Ecuador, por eje (ordenado por razón de brutos)'))
display(eje_ecu)

p = PAISES['Honduras']
brutos_oe = pd.Series(p['brutos_aut']).groupby(lambda tid: oe_de_eje(tid)).sum()
t = pd.DataFrame({'Brutos manual (PDF)': pd.Series(BRUTOS_MANUAL['Honduras']),
                  'Brutos pipeline (suma de ejes)': brutos_oe,
                  'Final manual': p['man']._oe.value_counts(),
                  'Final pipeline': p['aut']._oe.value_counts()})
t.loc['Total'] = t.sum()
t['Razón brutos'] = (t['Brutos pipeline (suma de ejes)'] / t['Brutos manual (PDF)']).round(2)
t['Razón final'] = (t['Final pipeline'] / t['Final manual']).round(2)
oe_hon = t
display(Markdown('#### Honduras, por objetivo estratégico'))
display(oe_hon)

#### Ecuador, por eje (ordenado por razón de brutos)

,Eje temático,Keywords pipeline,Brutos manual (PDF),Brutos pipeline,Razón brutos,Final manual,Final pipeline,Razón final
1121,Logística e infraestructura de transporte,34.0,362.0,2148.0,5.93,336.0,1000.0,2.98
1112,"Productividad, innovación y cadenas productivas",18.0,798.0,1481.0,1.86,698.0,1106.0,1.58
3113,Gestión de inversión y riesgos fiscales,16.0,129.0,218.0,1.69,116.0,150.0,1.29
1111,Entorno de negocios e inversión,29.0,3247.0,5332.0,1.64,2473.0,4228.0,1.71
3111,Gestión fiscal y macroeconomía,48.0,414.0,659.0,1.59,369.0,568.0,1.54
1122,Comercio exterior e integración comercial,22.0,417.0,640.0,1.53,381.0,477.0,1.25
1142,Financiamiento de vivienda,14.0,101.0,155.0,1.53,94.0,143.0,1.52
3231,Prevención del reclutamiento y violencia juvenil,36.0,89.0,135.0,1.52,83.0,107.0,1.29
1141,"Mercados financieros, financiamiento productivo e inclusión financiera",36.0,405.0,594.0,1.47,369.0,531.0,1.44
3221,Lavado de activos y flujos financieros ilícitos,20.0,150.0,220.0,1.47,126.0,159.0,1.26


#### Honduras, por objetivo estratégico

,Brutos manual (PDF),Brutos pipeline (suma de ejes),Final manual,Final pipeline,Razón brutos,Razón final
1.1,99,93,86,84,0.94,0.98
1.2,698,728,441,490,1.04,1.11
2.1,43,71,42,57,1.65,1.36
2.2,285,87,338,53,0.31,0.16
3.1,48,42,35,40,0.88,1.14
3.2,62,82,54,63,1.32,1.17
3.3,286,339,191,260,1.19,1.36
Total,1521,1442,1187,1047,0.95,0.88


### 3.3 Estudios del pipeline: banderas de diferencia metodológica
Cada estudio de la base automática se marca con estas banderas, usando el título + abstract normalizados:
- `pais_principal`: aparece el nombre del país ("ecuador", "honduras").
- `pais_solo_variantes`: no aparece el nombre del país, pero sí otra variante (ecuatoriano, Ecuadorian, hondureño…). La consulta manual solo usó el nombre, así que no habría encontrado estos estudios.
- `kw_exacta`: alguna palabra clave de los ejes donde apareció el estudio (según los respaldos crudos) está escrita como frase exacta. Si no hay ninguna, lo más probable es que haya entrado por el *stemming* (plurales, conjugaciones) o porque el texto indexado en OpenAlex es distinto del que tenemos. Es una aproximación.
- `reciente`: OpenAlex creó el registro (`created_date`) después de la fecha del reporte manual.

Las banderas se comparan entre los estudios que comparten ambas bases y los que solo trae el pipeline.

In [14]:
def banderas(p):
    aut, obras = p['aut'].copy(), p['obras']
    ejes_crudos = obras.ejes.reindex(aut['id']).values
    aut['pais_principal'] = aut._texto.map(lambda t: bool(p['re_pais'].search(t)))
    aut['pais_variante'] = aut._texto.map(lambda t: bool(p['re_pais_var'].search(t)))
    aut['pais_solo_variantes'] = ~aut.pais_principal & aut.pais_variante
    aut['kw_encontradas'] = [sorted({(tid, k) for tid in (ej if isinstance(ej, list) else e2) for k in p['re_kw'][tid].findall(t)})
                             for ej, e2, t in zip(ejes_crudos, aut._ejes, aut._texto)]
    aut['kw_exacta'] = aut.kw_encontradas.str.len() > 0
    aut['creado'] = obras.created_date.reindex(aut['id']).values
    aut['reciente'] = aut.creado > p['fecha_manual']
    aut['sin_abstract'] = aut.abstract.isna()
    return aut


def causa_solo_aut(r):
    if r.reciente:
        return '1. Registro creado en OpenAlex después del reporte manual'
    if r.pais_solo_variantes:
        return '2. Solo menciona una variante del país, no su nombre'
    if not r.pais_principal:
        return '3. Sin término de país en el título/abstract disponibles'
    if not r.kw_exacta:
        return '4. Sin palabra clave exacta (probable coincidencia por stemming)'
    return '5. Cumple país + palabra clave exacta (palabras clave distintas por eje, depuración manual u otras)'


BANDERAS = ['pais_principal', 'pais_solo_variantes', 'kw_exacta', 'reciente', 'sin_abstract']
for nombre, p in PAISES.items():
    b = banderas(p)
    p['banderas'] = b
    grupo = b.en_manual.map({True: 'Ambas', False: 'Solo automático'})
    tasas = (b.groupby(grupo)[BANDERAS].mean() * 100).round(1).T
    tasas['n Solo automático'] = b.loc[~b.en_manual, BANDERAS].sum()
    display(Markdown(f'#### {nombre}: % de estudios con cada bandera'))
    display(tasas)
    sa = b[~b.en_manual]
    causa = sa.apply(causa_solo_aut, axis=1).rename('causa')
    causas = causa.value_counts().sort_index()
    p['causas_solo_aut'] = causas
    display(Markdown(f'**{nombre}: causa principal de los {len(sa)} estudios que solo trae el pipeline** (se asigna la primera que se cumple)'))
    display(pd.DataFrame({'n': causas, '%': (causas / len(sa) * 100).round(1)}))
    display(Markdown(f'**{nombre}: estudios que solo trae el pipeline, por año y causa**'))
    display(pd.crosstab(causa.str[:2], sa.publication_year, margins=True, margins_name='Total'))

#### Honduras: % de estudios con cada bandera

en_manual,Ambas,Solo automático,n Solo automático
pais_principal,95.1,90.5,361
pais_solo_variantes,2.6,7.0,28
kw_exacta,86.4,88.5,353
reciente,0.2,39.3,157
sin_abstract,4.5,4.0,16


**Honduras: causa principal de los 399 estudios que solo trae el pipeline** (se asigna la primera que se cumple)

,n,%
causa,,
1. Registro creado en OpenAlex después del reporte manual,157,39.3
"2. Solo menciona una variante del país, no su nombre",20,5.0
3. Sin término de país en el título/abstract disponibles,8,2.0
4. Sin palabra clave exacta (probable coincidencia por stemming),25,6.3
"5. Cumple país + palabra clave exacta (palabras clave distintas por eje, depuración manual u otras)",189,47.4


**Honduras: estudios que solo trae el pipeline, por año y causa**

publication_year,2019,2020,2021,2022,2023,2024,2025,2026,Total
causa,,,,,,,,,
1.,3,0,3,0,13,17,21,100,157
2.,1,4,0,4,3,2,5,1,20
3.,0,1,3,2,0,0,1,1,8
4.,4,4,5,4,4,0,4,0,25
5.,18,21,20,30,22,24,29,25,189
Total,26,30,31,40,42,43,60,127,399


... (2 outputs left out)

**Ecuador: causa principal de los 6793 estudios que solo trae el pipeline** (se asigna la primera que se cumple)

,n,%
causa,,
1. Registro creado en OpenAlex después del reporte manual,1246,18.3
"2. Solo menciona una variante del país, no su nombre",1888,27.8
3. Sin término de país en el título/abstract disponibles,83,1.2
4. Sin palabra clave exacta (probable coincidencia por stemming),1356,20.0
"5. Cumple país + palabra clave exacta (palabras clave distintas por eje, depuración manual u otras)",2220,32.7


**Ecuador: estudios que solo trae el pipeline, por año y causa**

publication_year,2019,2020,2021,2022,2023,2024,2025,2026,Total
causa,,,,,,,,,
1.,54,43,51,68,107,128,319,476,1246
2.,204,140,122,124,134,208,428,528,1888
3.,7,3,10,3,10,14,14,22,83
4.,189,143,127,114,199,220,211,153,1356
5.,202,144,101,107,137,284,415,830,2220
Total,656,473,411,416,587,854,1387,2009,6793


**Palabras clave que más aportan estudios que solo trae el pipeline.** Para cada palabra clave encontrada como frase exacta se cuenta en cuántos estudios aparece, separando los de ambas bases y los que solo trae el pipeline. Un `% solo automático` alto indica términos que traen literatura que el manual no recogió: palabras clave nuevas o más amplias en el pipeline, o estudios que el manual depuró.

In [15]:
def tabla_keywords(p, ejes=None, top=15):
    b = p['banderas']
    x = b[['en_manual', 'kw_encontradas']].explode('kw_encontradas').dropna().reset_index(drop=True)
    x['eje'] = x.kw_encontradas.str[0]
    x['keyword'] = x.kw_encontradas.str[1]
    if ejes:
        x = x[x.eje.isin(ejes)]
    t = pd.crosstab([x.eje, x.keyword], x.en_manual.map({True: 'Ambas', False: 'Solo automático'}))
    t = t.reindex(columns=['Ambas', 'Solo automático'], fill_value=0)
    t['% solo automático'] = (t['Solo automático'] / t.sum(axis=1) * 100).round(1)
    return t.sort_values('Solo automático', ascending=False).head(top)


for nombre, p in PAISES.items():
    display(Markdown(f'#### {nombre}: top 15 palabras clave'))
    display(tabla_keywords(p))
display(Markdown('#### Ecuador: ejes con mayor brecha (1121 Logística y 1111 Entorno de negocios)'))
display(tabla_keywords(PAISES['Ecuador'], ejes=['1121', '1111'], top=20))

#### Honduras: top 15 palabras clave

en_manual                       Ambas  Solo automático  % solo automático
eje keyword                                                              
121 public health                  88               62               41.3
331 empleo                         46               60               56.6
    employment                     69               27               28.1
121 salud publica                  18               16               47.1
122 sistema educativo              16               12               42.9
210 politica fiscal                 3               12               80.0
121 health system                  22               10               31.2
120 indigenous people              17                9               34.6
331 desarrollo de competencias      5                9               64.3
122 educacion basica               18                8               30.8
121 sistema de salud                4                8               66.7
331 labor market                   27                8               22.9
123 social protection              11                7               38.9
122 education system               17                6               26.1
111 water and sanitation           15                6               28.6

#### Ecuador: top 15 palabras clave

en_manual                       Ambas  Solo automático  % solo automático
eje  keyword                                                             
1111 seguridad juridica           499              472               48.6
     competitividad               854              472               35.6
     marco legal                  448              226               33.5
2111 desarrollo de habilidades    386              218               36.1
3232 violencia de genero          320              216               40.3
3121 corrupcion                   341              198               36.7
1131 transformacion digital       292              191               39.5
1111 competitiveness              286              160               35.9
3121 contratacion publica         135              148               52.3
2111 formacion profesional        176              141               44.5
1111 legal framework              187              129               40.8
2131 agua potable                 166              114               40.7
1112 innovacion tecnologica       162              103               38.9
1131 brecha digital               215               99               31.5
3232 seguridad ciudadana          129               77               37.4

#### Ecuador: ejes con mayor brecha (1121 Logística y 1111 Entorno de negocios)

en_manual                          Ambas  Solo automático  % solo automático
eje  keyword                                                                
1111 competitividad                  854              472               35.6
     seguridad juridica              499              472               48.6
     marco legal                     448              226               33.5
     competitiveness                 286              160               35.9
     legal framework                 187              129               40.8
     legal certainty                  45               39               46.4
     inversion extranjera directa    118               32               21.3
1121 infraestructura vial             40               31               43.7
     puertos                          29               27               48.2
1111 entorno empresarial              61               21               25.6
     inversion privada                38               20               34.5
1121 road infrastructure              29               19               39.6
     costos logisticos                22               16               42.1
     sistema de transporte            27               15               35.7
1111 foreign direct investment        52               15               22.4
     business environment             27               15               35.7
1121 transporte de carga              15               14               48.3
     transporte urbano                18               12               40.0
     ports                            32               11               25.6
1111 private investment               26               10               27.8

### 3.4 Estudios del manual que no están en el pipeline: ¿en qué etapa se perdieron?
Cada estudio que solo está en el manual se busca en los respaldos crudos del pipeline, en este orden: llave (DOI u OpenAlex ID), DOI y título normalizado. Según el resultado, se clasifica así:
- **(a)** Ninguna consulta del pipeline lo recuperó.
- **(b)** Se recuperó, pero su subcampo no es de interés y lo eliminó el filtro.
- **(c)** Se recuperó y pasó el filtro, pero se eliminó como duplicado de otro registro.
- **(d)** No se recuperó y además está fuera del periodo 2019–2026 o tiene un tipo de documento no admitido.
- **(e)** Se recuperó y está en la base automática, pero con otro identificador o título (p. ej. OpenAlex fusionó registros).

En el grupo (a) se revisa, con el texto del manual, si tiene el nombre del país, una palabra clave exacta de su eje, y si su tópico cae en un subcampo de interés.

In [16]:
def ubicar_solo_manual(p, nombre):
    sm = p['cruce'][p['cruce'].aut_id.isna()].copy()
    obras = p['obras']
    por_oaid = {norm_oaid(i): i for i in obras.index}
    por_doi = {k: i for k, i in zip(obras._doi, obras.index) if ok(k)}
    por_tit = {k: i for k, i in zip(obras._titulo, obras.index) if ok(k)}
    por_llave = por_doi if nombre == 'Honduras' else por_oaid
    crudo, via = [], []
    for k, d, t in zip(sm._llave, sm._doi, sm._titulo):
        hit = next(((dic[v], n) for n, v, dic in (('llave', k, por_llave), ('doi', d, por_doi), ('título', t, por_tit))
                    if ok(v) and v in dic), (None, None))
        crudo.append(hit[0])
        via.append(hit[1])
    sm['crudo_id'], sm['via_crudo'] = crudo, via

    def etapa(r):
        if ok(r.crudo_id):
            o = obras.loc[r.crudo_id]
            if o.en_aut:
                return '(e) En la base automática con otro identificador o título'
            if not o.interes:
                return '(b) Recuperado; eliminado por el filtro de subcampo'
            return '(c) Recuperado; eliminado como duplicado de otro registro'
        if pd.isna(r._anio) or not 2019 <= r._anio <= 2026 or r[p['tipo']] not in TIPOS:
            return '(d) Fuera del periodo o del tipo de documento'
        return '(a) No recuperado por las consultas del pipeline'

    sm['etapa'] = sm.apply(etapa, axis=1)
    sm['pais_principal'] = sm._texto.map(lambda t: bool(p['re_pais'].search(t)))
    sm['pais_variante'] = sm._texto.map(lambda t: bool(p['re_pais_var'].search(t)))
    ejes_de = lambda r: r._ejes or [tid for tid in p['ejes'] if oe_de_eje(tid) == r._oe]
    sm['kw_exacta'] = [any(p['re_kw'][tid].search(r._texto) for tid in ejes_de(r)) for _, r in sm.iterrows()]
    sm['kw_exacta_cualquier_eje'] = sm._texto.map(lambda t: any(rx.search(t) for rx in p['re_kw'].values()))
    sub = sm['Tópico OpenAlex'].map(lambda s: p['topico_subcampo'].get(norm_text(s)))
    sm['topico_interes'] = sub.map(lambda s: 'sin tópico' if pd.isna(s) else ('sí' if s in p['interes'] else 'no'))
    sm['sin_abstract'] = sm[p['abstract']].isna()
    return sm


for nombre, p in PAISES.items():
    sm = ubicar_solo_manual(p, nombre)
    p['solo_manual'] = sm
    et = sm.etapa.value_counts().sort_index()
    display(Markdown(f'#### {nombre}: {len(sm)} filas que solo están en el manual'))
    display(pd.DataFrame({'n': et, '%': (et / len(sm) * 100).round(1)}))
    a = sm[sm.etapa.str.startswith('(a)')]
    if len(a):
        display(Markdown(f'**{nombre}, grupo (a): características de los {len(a)} no recuperados** (% del grupo)'))
        display((a[['pais_principal', 'pais_variante', 'kw_exacta', 'kw_exacta_cualquier_eje', 'sin_abstract']].mean() * 100).round(1).to_frame('%'))
        display(pd.crosstab(a.pais_principal, a.kw_exacta, margins=True, margins_name='Total',
                            rownames=['nombre del país en título/abstract'], colnames=['palabra clave exacta de su eje']))
        display(a.topico_interes.value_counts().rename('tópico en subcampo de interés').to_frame())

#### Honduras: 530 filas que solo están en el manual

,n,%
etapa,,
(a) No recuperado por las consultas del pipeline,395,74.5
(b) Recuperado; eliminado por el filtro de subcampo,109,20.6
(c) Recuperado; eliminado como duplicado de otro registro,4,0.8
(e) En la base automática con otro identificador o título,22,4.2


**Honduras, grupo (a): características de los 395 no recuperados** (% del grupo)

,%
pais_principal,73.7
pais_variante,12.9
kw_exacta,24.6
kw_exacta_cualquier_eje,24.6
sin_abstract,32.2


palabra clave exacta de su eje,False,True,Total
nombre del país en título/abstract,,,
False,95,9,104
True,203,88,291
Total,298,97,395


,tópico en subcampo de interés
topico_interes,
sí,307
no,62
sin tópico,26


... (2 outputs left out)

**Ecuador, grupo (a): características de los 409 no recuperados** (% del grupo)

,%
pais_principal,81.7
pais_variante,18.3
kw_exacta,73.1
kw_exacta_cualquier_eje,73.8
sin_abstract,29.3


palabra clave exacta de su eje,False,True,Total
nombre del país en título/abstract,,,
False,68,7,75
True,42,292,334
Total,110,299,409


,tópico en subcampo de interés
topico_interes,
sí,253
no,137
sin tópico,19


Revisión de 5 estudios del manual que no están en el pipeline, por país:

In [17]:
for nombre, p in PAISES.items():
    display(Markdown(f'**{nombre}**'))
    display(p['solo_manual'].sample(5, random_state=2)[[p['titulo'], '_anio', p['tipo'], 'Tópico OpenAlex', 'etapa', 'pais_principal', 'kw_exacta']])

**Honduras**

,Título,_anio,Tipo,Tópico OpenAlex,etapa,pais_principal,kw_exacta
579,Open Government in Honduras: Towards Effective Implementation,2023,book-chapter,Local Government Finance and Decentralization,(a) No recuperado por las consultas del pipeline,True,False
685,Energy potential from animal-origin biomass in Honduras,2026,book-chapter,Biofuel production and bioconversion,(a) No recuperado por las consultas del pipeline,True,False
736,"The distinctiveness of the European Union's influence on Latin America: European values, governance, and i...",2019,article,International Relations in Latin America,(a) No recuperado por las consultas del pipeline,True,False
747,Pathways to Decarbonize Honduras’ Power Sector,2025,article,Energy and Environment Impacts,(a) No recuperado por las consultas del pipeline,True,False
875,한·중미 5개국 FTA 발효에 따른 중미시장 활용방안(Trade and Investment Implications of Korea-Central America Free Trade Agreem...,2020,article,Asian Industrial and Economic Development,(a) No recuperado por las consultas del pipeline,False,False


**Ecuador**

,Título,_anio,Tipo,Tópico OpenAlex,etapa,pais_principal,kw_exacta
8260,Estudio de eficiencia energética de una bicicleta eléctrica urbana cargada con una estación de carga solar...,2023,article,Electric Vehicles and Infrastructure,(a) No recuperado por las consultas del pipeline,True,True
2782,Elaboración de un documental de memoria histórica sobre el carnaval en la provincia de Cañar,2019,dissertation,Literary and Cultural Studies,(b) Recuperado; eliminado por el filtro de subcampo,False,False
5828,Realidad aumentada como herramienta para la enseñanza del módulo formativo de sistemas eléctricos y electr...,2025,article,Experimental Learning in Engineering,(b) Recuperado; eliminado por el filtro de subcampo,True,True
7388,Minería de comentarios en la red social X sobre la percepción de la seguridad en la ciudadanía ecuatoriana,2025,article,"War, Law, and Justice",(b) Recuperado; eliminado por el filtro de subcampo,True,True
3015,"Transcripción de obras, en el género musical pasillo, del compositor lojano Dans Dagoberto Vilela",2020,dissertation,"Business, Innovation, and Economy",(b) Recuperado; eliminado por el filtro de subcampo,False,False


## 4. Conclusiones

Las cifras corresponden a la ejecución del notebook con los datos actuales. Las banderas de la sección 3.3 son aproximaciones: se calculan sobre el título y el abstract que tenemos, y OpenAlex indexa y tokeniza el texto a su manera.

### 4.1 Variables
- **Honduras:** 17 conceptos compartidos (13 exactos). Todas las columnas del manual tienen equivalente en el pipeline. El pipeline agrega 32 variables: jerarquía completa del tópico, ODS, detalle del acceso abierto, id/país/tipo/grupo de origen de las instituciones, editorial y trazabilidad de las imputaciones.
- **Ecuador:** 20 conceptos compartidos (17 exactos). Solo el manual trae `Concepto OpenAlex`, `Citations sum` (vacía) y su `ID` interno. El pipeline agrega 28 variables.
- **Diferencias de definición a tener en cuenta:**
  - Las "keywords" de los manuales son los *concepts* de OpenAlex.
  - El "ranking de pertinencia" del manual es una posición, no el `relevance_score`.
  - En el manual de Honduras, los ejes son banderas calculadas con una fórmula, no el resultado de la consulta.

### 4.2 Estudios compartidos y exclusivos

| | Honduras | Ecuador |
|---|---|---|
| Estudios del pipeline que también están en el manual | 648 de 1.047 (62 %) | 9.221 de 16.014 (58 %) |
| Filas del manual que cruzan con el pipeline | 657 de 1.187 (55 %) | 9.467 de 11.595 (82 %) |
| Filas solo en el manual | 530 | 2.128 |
| Estudios solo en el pipeline | 399 | 6.793 |

- **En los estudios que coinciden, los datos concuerdan:**

  | | Honduras | Ecuador |
  |---|---|---|
  | Mismo año | 98,6 % | 99,3 % |
  | Mismo objetivo estratégico | 92,4 % | 96,1 % |
  | Algún eje en común | 96 % | 96 % |

  Las citas del pipeline suelen ser iguales o mayores, por la fecha de extracción más reciente.
- **Los manuales conservan duplicados:**
  - Honduras: 54 filas sobrantes por título repetido (títulos de 4 palabras o más), 22 de ellas *datasets*.
  - Ecuador: 462 filas sobrantes por título (4 palabras o más) y 317 por OpenAlex ID repetido, casi todas artículos y tesis.
  - Con títulos de 4 palabras o más, el pipeline deja 0 títulos repetidos en Honduras y 1 en Ecuador ("T-EC 0023/CD 14941", que tiene 3 palabras en el original), una vez que su normalización de títulos ignora tildes y puntuación (nota de la sección 3.1).
- **La coincidencia real es algo mayor que la que da la regla principal.** En Ecuador, 327 filas con OpenAlex ID que no cruzan sí coinciden por título con 176 estudios del pipeline. Además, en la sección 3.4, 261 filas del manual aparecen en el pipeline con otro identificador. Ambas cosas indican que OpenAlex fusionó o cambió identificadores entre las dos extracciones.

### 4.3 Honduras
- **Estudios del manual que no están en el pipeline (530):**
  - 395 (75 %) no los recuperó ninguna consulta del pipeline.
  - 109 (21 %) se recuperaron, pero el filtro de subcampo los eliminó.
  - 22 están en el pipeline con otro DOI o título.
  - 4 se fusionaron como duplicados.
- **Los no recuperados se concentran en el OE 2.2** (261 de 395):
  - La búsqueda manual del 2.2 incluyó el término suelto "Resiliencia" / "resilience". 105 de los 395 contienen "resilien…".
  - El pipeline restringe ese término a "shock resilience" / "resiliencia ante choques".
  - Por eso el OE 2.2 tiene 338 estudios en el manual y 53 en el pipeline.
- **Otras dos causas de los no recuperados:**
  - El 26 % no tiene "Honduras" en el título ni en el abstract. El PDF no indica el campo de búsqueda, así que la búsqueda manual probablemente no se limitó a título/abstract.
  - El 75 % no contiene ninguna palabra clave del pipeline como frase exacta: las listas de palabras clave son distintas (el manual buscó por OE; el pipeline, por eje).
- **Estudios solo en el pipeline (399):**
  - El 39 % se creó en OpenAlex después del 1-jul-2026, casi todos publicados en 2026.
  - El 47 % cumple país + palabra clave exacta. Corresponden a diferencias de palabras clave o a la depuración manual.

### 4.4 Ecuador: ¿por qué el pipeline encuentra más estudios?
1. **Consultas más amplias desde el inicio.** Los brutos son 22.288 frente a 13.802 (×1,61), y son mayores en los 22 ejes. Las brechas más fuertes están en Logística e infraestructura de transporte (×5,9: 2.148 frente a 362) y en Productividad, innovación y cadenas productivas (×1,9).
2. **Variantes del nombre del país.** El 30,8 % de los estudios que solo trae el pipeline no menciona "Ecuador", sino "ecuatoriano/a", "Ecuadorian", etc. Entre los estudios de ambas bases esa proporción es 0,2 %, lo que confirma que la consulta manual, que solo usó "Ecuador", no los capta. Como causa principal explica 1.888 estudios (27,8 %).
3. **Stemming de las palabras clave.** El 24,7 % de los estudios que solo trae el pipeline no contiene ninguna palabra clave exacta, frente al 2,8 % en ambas bases. Como causa principal explica 1.356 estudios (20,0 %).
4. **Fecha de extracción.** 1.246 estudios (18,3 %) se crearon en OpenAlex después del 24-oct-2026, sobre todo publicaciones de 2025–2026.
5. **El resto, 2.220 estudios (32,7 %), cumple país + palabra clave exacta.** Corresponden a:
   - Diferencias en las listas de palabras clave por eje. Términos como "seguridad jurídica", "competitividad", "marco legal" o "contratación pública" aportan entre 34 % y 52 % de sus coincidencias en estudios que solo trae el pipeline.
   - La eliminación manual de tópicos.
   - La revisión manual de pertinencia.
6. **En la dirección contraria, el filtro de subcampo resta.** Eliminó 1.385 estudios del manual (65 % de los que solo están en el manual), sobre todo de los subcampos Tourism, Leisure and Hospitality Management, Marketing, Communication y Plant Science. Además, la deduplicación manual, más laxa (exigía DOI, autor, título y año iguales), infla el manual con 462 filas repetidas por título.